Based on: 

https://www.tensorflow.org/get_started/mnist/beginners 

https://www.oreilly.com/learning/not-another-mnist-tutorial-with-tensorflow

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# print the tensor flow version
print(tf.__version__)

In [ ]:
from utils import display_errors
from utils import unfold_labels
from dataset_helper import DataSetHelper

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))
print("Built with CUDA:", tf.test.is_built_with_cuda())

In [ ]:
num_classes = 10
learning_rate = 0.03
batch_size = 1024

In [ ]:
# setup tensorboard
import shutil

tensorboard_file_name = "./TensorFlowBoard/tensor_flow_basic"
shutil.rmtree(tensorboard_file_name, ignore_errors=True)  # start fresh each time
summary_writer = tf.summary.create_file_writer(tensorboard_file_name)

In [ ]:
#load mnist data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Keep original 28x28 images for display
x_test_images = x_test.copy()

# Preprocess: flatten and rescale to [0, 1]
x_train = x_train.reshape(-1, 784).astype(np.float32) / 255.0
x_test = x_test.reshape(-1, 784).astype(np.float32) / 255.0

# One-hot encode labels
y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_oh = tf.keras.utils.to_categorical(y_test, num_classes)

# Build tf.data.Dataset with shuffle, repeat, batch, and prefetch
train_dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train_oh.astype(np.float32))
).shuffle(x_train.shape[0]).repeat().batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
training_set_size = x_train.shape[0]
num_epochs = 20
steps_per_epoch = training_set_size // batch_size
epochs = num_epochs * steps_per_epoch  # total training steps

In [ ]:
# define the model variables
# z = W*x + b
# y = softmax(z)
W = tf.Variable(tf.random.uniform([784, 10]), name="W")
b = tf.Variable(tf.random.uniform([10]), name="b")

In [ ]:
optimizer = tf.optimizers.SGD(learning_rate=learning_rate)

@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        z = tf.add(tf.matmul(x_batch, W), b)
        loss = tf.reduce_mean(
            tf.nn.softmax_cross_entropy_with_logits(labels=y_batch, logits=z))
    gradients = tape.gradient(loss, [W, b])
    optimizer.apply_gradients(zip(gradients, [W, b]))
    y_pred = tf.nn.softmax(z)
    correct = tf.equal(tf.argmax(y_pred, 1), tf.argmax(y_batch, 1))
    accuracy = tf.reduce_mean(tf.cast(correct, tf.float32))
    return loss, accuracy

def predict(x_data):
    z = tf.add(tf.matmul(x_data, W), b)
    return tf.argmax(tf.nn.softmax(z), 1)

def evaluate(x_data, y_data):
    z = tf.add(tf.matmul(x_data, W), b)
    y_pred = tf.nn.softmax(z)
    correct = tf.equal(tf.argmax(y_pred, 1), tf.argmax(y_data, 1))
    return tf.reduce_mean(tf.cast(correct, tf.float32))

In [ ]:
loss_batch = []
accuracy_batch = []

In [ ]:
for index, (batch_xs, batch_ys) in enumerate(train_dataset.take(epochs)):
    train_loss, train_accuracy = train_step(batch_xs, batch_ys)

    if index % (epochs // 100) == 0:
        loss_batch.append(train_loss.numpy())
        accuracy_batch.append(train_accuracy.numpy())

    if index % (epochs // 10) == 0:
        print(f"Batch: {index}; Accuracy: {train_accuracy.numpy():.4f}; Loss: {train_loss.numpy():.4f}")

pred = predict(tf.constant(x_test, dtype=tf.float32)).numpy()
test_accuracy = evaluate(
    tf.constant(x_test, dtype=tf.float32),
    tf.constant(y_test_oh, dtype=tf.float32))
print(f"Accuracy: {test_accuracy.numpy():.4f}")

In [ ]:
plt.plot(range(0, len(loss_batch)), loss_batch, 'r-', label='Batch Loss')
plt.plot(range(0, len(accuracy_batch)), accuracy_batch, 'b-', label='Batch Accuracy')
plt.legend(loc='upper right')
plt.title('Training accuracy and loss')
plt.show()

In [ ]:
labels = y_test
conf = tf.math.confusion_matrix(labels, pred, num_classes=10)
print(conf.numpy())

In [ ]:
display_errors(x_test_images, y_test, pred, grid = 8)